# 07 · Aplicação na Olist e validação com o atraso real

O melhor encoder (treinado no RePro) prevê os aspectos de todos os comentários da Olist. A predição de **entrega negativa** é comparada com o **atraso registrado no sistema**, uma verdade de campo que não depende de anotação humana nem de IA.

Ambiente: **GPU T4**.

In [ ]:
# Configuração: no Colab, clona o repositório, instala o pacote `absa`
# e guarda cache do LLM, modelos e predições no Drive (sobrevive à queda da sessão)
import sys
if "google.colab" in sys.modules:
    !git clone -q https://github.com/vnevesl/pln-absa-olist.git 2>/dev/null || git -C pln-absa-olist pull -q
    %cd pln-absa-olist
    !pip -q install -e ".[transformers]"
    from google.colab import drive
    drive.mount("/content/drive")
    !mkdir -p /content/drive/MyDrive/pln-absa-olist/llm_cache /content/drive/MyDrive/pln-absa-olist/models /content/drive/MyDrive/pln-absa-olist/interim
    !ln -sfn /content/drive/MyDrive/pln-absa-olist/llm_cache data/llm_cache
    !ln -sfn /content/drive/MyDrive/pln-absa-olist/models models
    !ln -sfn /content/drive/MyDrive/pln-absa-olist/interim data/interim

In [ ]:
import numpy as np
import pandas as pd
from absa.config import ASPECTS, INTERIM, LABEL_COLUMNS, MODELS, POLARITIES
from absa.data import load_reviews
from absa.metrics import delivery_validation, per_aspect_report, tuple_f1
from absa.models.encoder import ENCODERS, EncoderAspectClassifier, TrainConfig
from absa.preprocess import clean_light

MELHOR = "bertimbau_humano"  # ajuste conforme o notebook 06
clf = EncoderAspectClassifier(TrainConfig(model_name=ENCODERS[MELHOR.split("_")[0]])).load(MODELS / f"{MELHOR}.pt")

olist = load_reviews()
proba = clf.predict_proba(olist.texto.map(clean_light).tolist())
j, k = ASPECTS.index("entrega"), POLARITIES.index("negativo")
olist["p_entrega_neg"] = proba[:, j, k]
for a_idx, a in enumerate(ASPECTS):
    olist[f"asp_{a}"] = [POLARITIES[i] for i in proba[:, a_idx].argmax(axis=1)]
olist.to_csv(INTERIM / "olist_predicoes.csv.gz", index=False)

## A predição de "entrega negativa" acompanha o atraso real?

In [ ]:
ent = olist[olist.entregue]
print(f"AUC (P(entrega negativa) × atrasado): {delivery_validation(ent.atrasado, ent.p_entrega_neg):.3f}")
pd.crosstab(ent.atrasado, ent.asp_entrega, normalize="index").mul(100).round(1)

In [ ]:
faixas = pd.cut(ent.atraso_dias, [-60, -10, -1, 0, 3, 7, 15, 200])
ent.groupby(faixas, observed=True).asp_entrega.apply(lambda s: 100 * (s == "negativo").mean()).round(1)

## O que explica a nota baixa? (aspectos negativos como variáveis)

In [ ]:
from sklearn.linear_model import LogisticRegression
X = pd.DataFrame({a: (olist[f"asp_{a}"] == "negativo").astype(int) for a in ASPECTS})
y = (olist.nota <= 2).astype(int)
reg = LogisticRegression().fit(X, y)
pd.Series(np.exp(reg.coef_[0]), index=ASPECTS, name="razao_de_chances").sort_values(ascending=False).round(2)

## Mapa de reclamações: aspecto negativo por categoria e região

In [ ]:
neg = olist.assign(**{a: olist[f"asp_{a}"] == "negativo" for a in ASPECTS})
top_cat = olist.categoria.value_counts().head(12).index
print(neg[neg.categoria.isin(top_cat)].groupby("categoria")[ASPECTS].mean().mul(100).round(1))
neg.groupby("regiao")[ASPECTS].mean().mul(100).round(1)

## Amostra anotada da Olist (opcional)

In [ ]:
from absa.config import ANNOTATED
if (ANNOTATED / "olist_anotado.csv").exists():
    from absa.sampling import load_olist_gold
    gold = load_olist_gold()
    pred = olist.set_index("review_id").loc[gold.review_id, LABEL_COLUMNS].reset_index(drop=True)
    print("F1 tupla na Olist:", round(tuple_f1(gold, pred), 3))
    display(per_aspect_report(gold, pred).round(3))